# Part 3: Intelligence Layers

RAG with LCEL, CAG with a semantic cache and CRAG with corrective retrieval.

In [ ]:
from prime_land.chat_service.cag_cache import CAGCache
from prime_land.chat_service.cag_service import CAGService
from prime_land.chat_service.crag_service import CRAGService
from prime_land.chat_service.rag_service import RAGService
from prime_land.config import CONFIG, get_path, load_faqs
from prime_land.ingest_document.indexer import get_client, open_store
from prime_land.providers import get_embeddings, get_llm

strategy = CONFIG["retrieval"]["strategy"]
embeddings = get_embeddings()
llm = get_llm()
client = get_client()
store = open_store(client, embeddings, strategy)
rag = RAGService(store, embeddings, llm, strategy=strategy)

## RAG

In [ ]:
result = rag.generate("Which gated house community is available in Kalutara?")
print(result["answer"])
print(result["evidence_urls"])

## CAG

In [ ]:
cache = CAGCache(embeddings, get_path("cache_dir"), CONFIG["cag"]["similarity_threshold"], CONFIG["cag"]["history_ttl_hours"])
cag = CAGService(rag, cache)
print(cag.warm_faqs(load_faqs()), "FAQs warmed")

In [ ]:
first = cag.generate("What lands are available at Prime Lands?")
second = cag.generate("Tell me what lands Prime Lands has")
for result in (first, second):
    print(result["cache_hit"], result["cache_source"], round(result["similarity"], 3), round(result["generation_time"], 3))
print(cag.stats())

## CRAG

In [ ]:
crag = CRAGService(rag, llm)
for question in ["Which gated house community is available in Kalutara?", "I want a quiet land by the beach in the south"]:
    result = crag.generate(question)
    print(question)
    print(round(result["confidence_initial"], 3), "->", round(result["confidence_final"], 3), "corrected:", result["correction_applied"])

In [ ]:
client.close()